# Task 4 — Predictive & Forecasting Dashboard
## Advanced Data Visualization (COMP-834) | PAK-AUSTRIA FACHHOCHSCHULE
### World Bank GDP Per Capita: Machine Learning Forecasting
**Data Source:** World Bank API — `NY.GDP.PCAP.CD` (GDP per capita, current USD)  
**Countries:** USA, China, India, UK, Germany, France, Japan, Pakistan, Brazil, South Africa  
**Period:** 2000–2023 (Actual) + 2024–2028 (Forecasted)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

## 1. Data Import & Overview

In [ ]:
# Load cleaned World Bank dataset
df = pd.read_csv('worldbank_gdp_clean.csv')
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nCountries:", df['Country'].unique())
print("\nYear Range:", df['Year'].min(), "to", df['Year'].max())
df.head(10)

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics
print("Summary Statistics:")
print(df.groupby('Country')['GDP_Per_Capita_USD'].agg(['min','max','mean']).round(0))

In [ ]:
# Visualize GDP trends
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('World Bank GDP Per Capita Analysis (2000-2023)', fontsize=16, fontweight='bold')

# GDP trend lines
ax = axes[0,0]
for country in df['Country'].unique():
    cdf = df[df['Country'] == country]
    ax.plot(cdf['Year'], cdf['GDP_Per_Capita_USD'], marker='o', markersize=2, label=country)
ax.set_title('GDP Per Capita Trends')
ax.set_xlabel('Year'); ax.set_ylabel('USD')
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)

# Growth rate heatmap
pivot = df.pivot(index='Country', columns='Year', values='Growth_Rate_Pct').fillna(0)
im = axes[0,1].imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=-10, vmax=10)
axes[0,1].set_xticks(range(len(pivot.columns)))
axes[0,1].set_xticklabels(pivot.columns, rotation=90, fontsize=8)
axes[0,1].set_yticks(range(len(pivot.index)))
axes[0,1].set_yticklabels(pivot.index, fontsize=8)
axes[0,1].set_title('Annual Growth Rate Heatmap (%)')
plt.colorbar(im, ax=axes[0,1])

# 2023 GDP bar chart
df_2023 = df[df['Year'] == 2023].sort_values('GDP_Per_Capita_USD', ascending=True)
axes[1,0].barh(df_2023['Country'], df_2023['GDP_Per_Capita_USD'], color='steelblue')
axes[1,0].set_title('GDP Per Capita 2023 (USD)')
axes[1,0].set_xlabel('USD')
for i, v in enumerate(df_2023['GDP_Per_Capita_USD']):
    axes[1,0].text(v + 200, i, f'${v:,.0f}', va='center', fontsize=8)

# Region pie
region_2023 = df[df['Year'] == 2023].groupby('Region')['GDP_Per_Capita_USD'].sum()
axes[1,1].pie(region_2023.values, labels=region_2023.index, autopct='%1.1f%%', startangle=90)
axes[1,1].set_title('GDP Share by Region (2023)')

plt.tight_layout()
plt.savefig('eda_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print("EDA charts saved.")

## 3. Machine Learning — Linear Regression
Linear Regression models the long-term linear trend of GDP growth using year as feature.
- **Feature:** Year (encoded as integer)
- **Target:** GDP Per Capita (USD)
- **Purpose:** Trend prediction, baseline model

In [ ]:
# Linear Regression per country
lr_results = {}
future_years = np.array([2024, 2025, 2026, 2027, 2028])
all_years = np.array(list(range(2000, 2024)))

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()
fig.suptitle('Linear Regression — GDP Per Capita Predictions', fontsize=14, fontweight='bold')

for i, country in enumerate(sorted(df['Country'].unique())):
    cdf = df[df['Country'] == country].sort_values('Year')
    X = cdf['Year'].values.reshape(-1, 1)
    y = cdf['GDP_Per_Capita_USD'].values
    
    lr = LinearRegression()
    lr.fit(X, y)
    y_pred = lr.predict(X)
    future_pred = lr.predict(future_years.reshape(-1,1))
    
    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    lr_results[country] = {'MAE': round(mae,2), 'RMSE': round(rmse,2), 'R2': round(r2,4), 'model': lr}
    
    ax = axes[i]
    ax.plot(cdf['Year'], y, 'b-o', markersize=3, label='Actual', linewidth=1.5)
    ax.plot(cdf['Year'], y_pred, 'r--', label='LR Fit', linewidth=1.5)
    ax.plot(future_years, future_pred, 'g-^', markersize=5, label='Forecast', linewidth=1.5)
    ax.axvline(x=2023.5, color='gray', linestyle=':', alpha=0.7)
    ax.set_title(f'{country}\nR²={r2:.3f}', fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
    ax.set_xlabel('Year', fontsize=8); ax.set_ylabel('USD', fontsize=8)

plt.tight_layout()
plt.savefig('lr_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

# Print metrics table
metrics_df = pd.DataFrame(lr_results).T[['MAE','RMSE','R2']]
print("\nLinear Regression Metrics:")
print(metrics_df.to_string())

## 4. Machine Learning — ARIMA Forecasting
ARIMA (AutoRegressive Integrated Moving Average) is a time-series model ideal for economic data.
- **Order:** (1, 1, 1) — AR(1) + first differencing + MA(1)
- **Advantage:** Captures autocorrelation and trend patterns
- **Use case:** Economic and price forecasting

In [ ]:
# ARIMA per country
arima_results = {}

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()
fig.suptitle('ARIMA(1,1,1) — GDP Per Capita Forecasting', fontsize=14, fontweight='bold')

for i, country in enumerate(sorted(df['Country'].unique())):
    cdf = df[df['Country'] == country].sort_values('Year')
    y = cdf['GDP_Per_Capita_USD'].values
    
    model = ARIMA(y, order=(1,1,1))
    fitted = model.fit()
    y_fitted = fitted.fittedvalues
    forecast = fitted.forecast(steps=5)
    
    mae = mean_absolute_error(y[1:], y_fitted[1:])
    rmse = np.sqrt(mean_squared_error(y[1:], y_fitted[1:]))
    r2 = r2_score(y[1:], y_fitted[1:])
    arima_results[country] = {'MAE': round(mae,2), 'RMSE': round(rmse,2), 'R2': round(r2,4)}
    
    ax = axes[i]
    ax.plot(cdf['Year'], y, 'b-o', markersize=3, label='Actual', linewidth=1.5)
    ax.plot(cdf['Year'][1:], y_fitted[1:], 'r--', label='ARIMA Fit', linewidth=1.5)
    ax.plot(future_years, forecast, 'g-^', markersize=5, label='Forecast', linewidth=1.5)
    ax.axvline(x=2023.5, color='gray', linestyle=':', alpha=0.7)
    ax.set_title(f'{country}\nR²={r2:.3f}', fontsize=9)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
    ax.set_xlabel('Year', fontsize=8); ax.set_ylabel('USD', fontsize=8)

plt.tight_layout()
plt.savefig('arima_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

arima_df = pd.DataFrame(arima_results).T[['MAE','RMSE','R2']]
print("\nARIMA Metrics:")
print(arima_df.to_string())

## 5. Model Comparison & Evaluation

In [ ]:
# Side-by-side comparison
lr_m = pd.DataFrame(lr_results).T[['MAE','RMSE','R2']].rename(columns={'MAE':'LR_MAE','RMSE':'LR_RMSE','R2':'LR_R2'})
ar_m = pd.DataFrame(arima_results).T[['MAE','RMSE','R2']].rename(columns={'MAE':'ARIMA_MAE','RMSE':'ARIMA_RMSE','R2':'ARIMA_R2'})
compare = lr_m.join(ar_m)
compare['Better_Model'] = compare.apply(lambda r: 'ARIMA' if r['ARIMA_R2'] > r['LR_R2'] else 'Linear Reg.', axis=1)
print("Model Comparison (R² Score):")
print(compare[['LR_R2','ARIMA_R2','Better_Model']].to_string())

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(compare))
w = 0.35
axes[0].bar(x-w/2, compare['LR_R2'], w, label='Linear Regression', color='steelblue')
axes[0].bar(x+w/2, compare['ARIMA_R2'], w, label='ARIMA', color='darkorange')
axes[0].set_xticks(x); axes[0].set_xticklabels(compare.index, rotation=45, ha='right')
axes[0].set_title('R² Score Comparison'); axes[0].legend(); axes[0].set_ylim(0, 1.1)
axes[0].axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='Good threshold')
axes[0].grid(alpha=0.3)

axes[1].bar(x-w/2, compare['LR_RMSE'], w, label='Linear Regression', color='steelblue')
axes[1].bar(x+w/2, compare['ARIMA_RMSE'], w, label='ARIMA', color='darkorange')
axes[1].set_xticks(x); axes[1].set_xticklabels(compare.index, rotation=45, ha='right')
axes[1].set_title('RMSE Comparison (Lower is Better)'); axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Export Forecast Dataset for Power BI

In [ ]:
forecast_df = pd.read_csv('worldbank_forecast.csv')
print("Forecast Dataset Shape:", forecast_df.shape)
print("\nSample Forecast (United States):")
print(forecast_df[forecast_df['Country']=='United States'][['Year','GDP_Per_Capita_USD','LR_Predicted','Type']].to_string())
print("\n✅ Files ready for Power BI import:")
print("  1. worldbank_gdp_clean.csv  → Task 3 dashboard")
print("  2. worldbank_forecast.csv   → Task 4 predictive dashboard")
print("  3. model_metrics.csv        → Error analysis table")